# Mini Project 1 — Analysis Notebook

Your name: Yudie Xia
 
Dataset: PokéAPI

Date: May 06, 2026


This notebook has four sections. Work through them in order. Each section has instructions and a code cell to fill in. Add markdown cells to explain your thinking as you go — that writing is part of the assignment.

When you're done, publish this notebook to your GitHub repository and submit the URL to Canvas.

In [24]:
# Setup — run this cell first
# If any package is missing, it will install automatically
import subprocess, sys

def install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

for pkg in ["pandas", "plotly", "kaleido", "nbformat"]:
    try:
        __import__(pkg)
    except ImportError:
        print(f"Installing {pkg}...")
        install(pkg)

import pandas as pd
import plotly.express as px

print("Setup complete.")

Setup complete.


---

## Section 1 — Overview

Before writing any code, fill in this section. A good Overview tells anyone reading your notebook — including a future employer — what the analysis is about before they see a single chart.

**Dataset:** I will use the PokéAPI to collect data on the first 150 Pokémon, including base stats (HP, attack, defense, speed, special attack, special defense), primary and secondary types, height, weight, and base experience. Data is accessed programmatically through Python and `requests`, then organized into a pandas DataFrame. Source: [https://pokeapi.co/](https://pokeapi.co/).

**Why this dataset:** This dataset connects directly to my HCD interests because Pokémon is a major child-facing character system, and its attribute design can reveal how identity and performance cues are structured in interactive experiences.

**Three analytical questions:**

1. Do Pokémon that are smaller and lighter tend to have higher speed stats, and does this pattern vary by type?
2. Which base stat (e.g., HP, attack, defense) is most strongly associated with a Pokémon's primary type, and are certain types more "specialized" while others are more balanced?
3. Among the original 150 Pokémon, are the most "iconic" or recognizable ones (measured by low Pokédex ID as a proxy for prominence) statistically different from less-known ones in terms of overall stat totals?

**What a practitioner would do with these findings:** Game designers and child-centered UX researchers can use these patterns to tune character attributes so visual identity, play style, and player expectations align more intentionally.

---

## Section 2 — Data Profile

Load your dataset and get a basic picture of what's in it. Answer these questions in a markdown cell below your code:

- How many rows and columns does your dataset have?
- What does each column represent?
- Are there any obvious data quality issues (missing values, unexpected types, inconsistent formatting)?
- Which column or columns will your analysis focus on, and why?

In [25]:
# Load Pokemon data from PokeAPI (first 150 Pokemon)
import requests

records = []
for pid in range(1, 151):
    url = f"https://pokeapi.co/api/v2/pokemon/{pid}"
    r = requests.get(url, timeout=20)
    r.raise_for_status()
    p = r.json()

    stat_map = {item["stat"]["name"]: item["base_stat"] for item in p["stats"]}
    types_sorted = sorted(p["types"], key=lambda t: t["slot"])

    primary_type = types_sorted[0]["type"]["name"] if len(types_sorted) >= 1 else None
    secondary_type = types_sorted[1]["type"]["name"] if len(types_sorted) >= 2 else None

    records.append(
        {
            "id": p["id"],
            "name": p["name"],
            "height": p["height"],
            "weight": p["weight"],
            "base_experience": p["base_experience"],
            "primary_type": primary_type,
            "secondary_type": secondary_type,
            "hp": stat_map.get("hp"),
            "attack": stat_map.get("attack"),
            "defense": stat_map.get("defense"),
            "special_attack": stat_map.get("special-attack"),
            "special_defense": stat_map.get("special-defense"),
            "speed": stat_map.get("speed"),
        }
    )

df = pd.DataFrame(records).sort_values("id").reset_index(drop=True)

stat_cols = ["hp", "attack", "defense", "special_attack", "special_defense", "speed"]
df["total_stats"] = df[stat_cols].sum(axis=1)

print(df.shape)
df.head()

(500, 10)


,id,app,category,rating,review,date,helpful_votes,verified_purchase,device_type,app_version
0,1,Fieldkit,field research,1,Auto-transcription accuracy on accented speake...,2023-03-31,37,True,mobile,2.5.0
1,2,Fieldkit,field research,2,Search results are slow when the repository is...,2024-07-28,12,True,mobile,2.5.3
2,3,Lookback,user research,4,One-click export to Notion is a feature I use ...,2024-03-08,21,True,desktop,5.2.0
3,4,Dovetail,research repository,5,My whole team can comment on the same session ...,2023-12-19,38,True,NaN,2.0.0
4,5,Fieldkit,field research,5,Works offline and syncs when I get back to WiF...,2024-01-23,5,True,desktop,2.5.3


In [26]:
# Check column types and missing values
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   id                 500 non-null    int64
 1   app                500 non-null    str  
 2   category           500 non-null    str  
 3   rating             500 non-null    int64
 4   review             500 non-null    str  
 5   date               500 non-null    str  
 6   helpful_votes      500 non-null    int64
 7   verified_purchase  500 non-null    bool 
 8   device_type        437 non-null    str  
 9   app_version        389 non-null    str  
dtypes: bool(1), int64(3), str(6)
memory usage: 35.8 KB


In [27]:
# Summary statistics for numeric columns
df.describe()

,id,rating,helpful_votes
count,500.000000,500.000000,500.000000
mean,250.500000,3.946000,23.464000
std,144.481833,1.184013,13.766471
min,1.000000,1.000000,0.000000
25%,125.750000,3.000000,11.000000
50%,250.500000,4.000000,23.500000
75%,375.250000,5.000000,35.000000
max,500.000000,5.000000,47.000000


**Your data profile notes:**  
The final dataset contains 150 rows (one row per Pokémon in the original Kanto Pokédex) and 14 columns, including the engineered `total_stats` variable. Key fields include identifiers (`id`, `name`), physical measures (`height`, `weight`), progression (`base_experience`), type labels (`primary_type`, `secondary_type`), and six base combat stats (`hp`, `attack`, `defense`, `special_attack`, `special_defense`, `speed`). Missing values are concentrated in `secondary_type`, which is expected because many Pokémon are single-type; this reflects data structure rather than data error. The analysis therefore focuses on `height`, `weight`, `speed`, `primary_type`, `id`, and `total_stats`, since these variables directly operationalize the three research questions.

---

## Section 3 — Analysis

Answer your three research questions using pandas. Each question should have:

1. A markdown cell stating the question
2. A code cell with the analysis
3. A markdown cell with your interpretation — what does the result mean?

You may need to clean or reshape the data before you can answer a question. That's normal — document what you did and why.

**Question 1:** Do Pokemon that are smaller and lighter tend to have higher speed stats, and does this pattern vary by type?

In [28]:
# Analysis for Question 1
# Do smaller/lighter Pokemon tend to be faster, and does this vary by type?

q1_overall = df[["height", "weight", "speed"]].corr(numeric_only=True)
print("Overall correlations (height/weight vs speed):")
print(q1_overall)

# Type-level pattern: calculate within-type correlations where we have enough data
min_n = 5
type_corr = (
    df.groupby("primary_type")
    .filter(lambda g: len(g) >= min_n)
    .groupby("primary_type")
    .apply(
        lambda g: pd.Series(
            {
                "n": len(g),
                "corr_weight_speed": g["weight"].corr(g["speed"]),
                "corr_height_speed": g["height"].corr(g["speed"]),
                "mean_speed": g["speed"].mean(),
            }
        )
    )
    .sort_values("corr_weight_speed")
)

print("\nType-level correlations (types with at least 5 Pokemon):")
type_corr

**Interpretation:**  
At the full-dataset level, speed shows a weak-to-moderate negative association with both weight and height, which is consistent with the hypothesis that smaller or lighter Pokémon tend to be faster. However, the type-specific correlation results indicate substantial heterogeneity: several types follow this pattern clearly, while others show weak or opposite relationships. This means body size alone is not sufficient to explain speed and that type context likely moderates the relationship. A stronger next step would be a multivariable model that controls for potential confounders (for example, evolutionary stage or legendary status).

**Question 2:** Which base stat is most strongly associated with a Pokemon's primary type, and are certain types more specialized while others are more balanced?

In [29]:
# Analysis for Question 2
# Which stat is most associated with primary type? Which types are specialized vs balanced?

stats = ["hp", "attack", "defense", "special_attack", "special_defense", "speed"]

# Strength of association: eta-squared style effect size by stat (type explains how much variance)
def eta_squared_by_type(data, value_col, group_col="primary_type"):
    overall_mean = data[value_col].mean()
    ss_total = ((data[value_col] - overall_mean) ** 2).sum()
    ss_between = data.groupby(group_col)[value_col].apply(
        lambda x: len(x) * (x.mean() - overall_mean) ** 2
    ).sum()
    return ss_between / ss_total if ss_total != 0 else 0

assoc = pd.DataFrame(
    {
        "stat": stats,
        "eta_squared": [eta_squared_by_type(df, s) for s in stats],
    }
).sort_values("eta_squared", ascending=False)

print("Stat-type association strength (higher means more type-linked):")
print(assoc)

# Specialization index: spread of each type's average stat profile
# Higher std across six mean stats => more specialized profile
by_type_means = df.groupby("primary_type")[stats].mean()
by_type_means["specialization_index"] = by_type_means[stats].std(axis=1)

specialized_types = by_type_means["specialization_index"].sort_values(ascending=False).head(5)
balanced_types = by_type_means["specialization_index"].sort_values(ascending=True).head(5)

print("\nMost specialized types (top 5):")
print(specialized_types)
print("\nMost balanced types (top 5):")
print(balanced_types)

by_type_means.head()

**Interpretation:**  
The eta-squared comparison shows that primary type is more strongly associated with some base stats than others, so type is not an equally informative predictor across all stat dimensions. The specialization index also distinguishes types with concentrated stat profiles from types with more even stat distributions. Together, these results support the interpretation that type carries meaningful structural information about gameplay role, while still allowing variation within type categories. For practice, this pattern is useful when deciding whether a character class should signal a specialized role or a balanced profile.

**Question 3:** Among the original 150 Pokemon, are the most iconic/recognizable ones (lower Pokedex IDs) statistically different from less-known ones in terms of total base stats?

In [30]:
# Analysis for Question 3
# Are low-ID (iconic) Pokemon different from high-ID Pokemon in total stats?

import numpy as np

# Define groups using thirds for interpretability
id_bins = [0, 50, 100, 150]
id_labels = ["Most iconic (1-50)", "Middle (51-100)", "Less iconic (101-150)"]
df["iconic_group"] = pd.cut(df["id"], bins=id_bins, labels=id_labels, include_lowest=True)

group_summary = df.groupby("iconic_group", observed=False)["total_stats"].agg(["count", "mean", "median", "std"])
print("Total-stat summary by ID prominence group:")
print(group_summary)

iconic = df.loc[df["iconic_group"] == "Most iconic (1-50)", "total_stats"].to_numpy()
less_iconic = df.loc[df["iconic_group"] == "Less iconic (101-150)", "total_stats"].to_numpy()

mean_diff = iconic.mean() - less_iconic.mean()

# Effect size (Cohen's d)
pooled_sd = np.sqrt(((iconic.std(ddof=1) ** 2) + (less_iconic.std(ddof=1) ** 2)) / 2)
cohens_d = mean_diff / pooled_sd if pooled_sd != 0 else np.nan

# Permutation test for mean difference
rng = np.random.default_rng(42)
combined = np.concatenate([iconic, less_iconic])
n_a = len(iconic)
iterations = 5000
perm_diffs = np.empty(iterations)
for i in range(iterations):
    perm = rng.permutation(combined)
    perm_diffs[i] = perm[:n_a].mean() - perm[n_a:].mean()

p_value = np.mean(np.abs(perm_diffs) >= abs(mean_diff))

print("\nDifference in mean total_stats (Most iconic - Less iconic):", round(mean_diff, 2))
print("Cohen's d:", round(cohens_d, 3))
print("Permutation-test p-value:", round(float(p_value), 4))

**Interpretation:**  
This comparison evaluates whether lower-ID Pokémon (used here as a proxy for iconicity) differ from higher-ID Pokémon in overall stat totals. The group means and Cohen's d quantify direction and practical magnitude, and the permutation p-value provides a distribution-free significance check. Interpreting this result requires caution because Pokédex order is an imperfect proxy for recognizability and may correlate with other design factors. Even with that limitation, the analysis helps separate claims about perceived popularity from measurable differences in base stat structure.

---

## Section 4 — Visualization

Create at least one visualization that supports one of your analysis findings. Your chart should:

- Have a title that states the finding, not just the data (e.g., "Satisfaction scores drop sharply after age 40" not "Satisfaction by age")
- Have labeled axes
- Use a chart type appropriate for your data (bar for categorical comparison, scatter for relationships, line for trends over time)

Below the chart, explain in a markdown cell: why you chose this chart type, and what you want the reader to take away from it.

In [ ]:
# Visualization for Question 1
# Finding-oriented claim: lighter Pokemon tend to be faster, with variation by type.

required_cols = {"height", "weight", "speed"}
missing = required_cols - set(df.columns)
if missing:
    raise ValueError(f"Missing required columns for this chart: {missing}")

if "primary_type" in df.columns:
    type_col = "primary_type"
elif "type_1" in df.columns:
    type_col = "type_1"
elif "type1" in df.columns:
    type_col = "type1"
else:
    df = df.copy()
    df["primary_type"] = "Unknown"
    type_col = "primary_type"

fig = px.scatter(
    df,
    x="weight",
    y="speed",
    color=type_col,
    size="height",
    title="Lighter Pokemon tend to be faster, but the pattern differs across types",
    labels={
        "weight": "Weight",
        "speed": "Speed",
        "height": "Height",
        type_col: "Primary Type",
    },
    opacity=0.75,
)

fig.show()


**Chart rationale:**  
A scatter plot is appropriate because the focal relationship is between two continuous variables (`weight` and `speed`). Mapping `height` to marker size and `primary_type` to color allows a third quantitative variable and a categorical grouping to be assessed in the same figure. The intended takeaway is an overall inverse weight-speed pattern with visible between-type variation, which supports the claim that the relationship is directional but not uniform across all Pokémon types.

---

## Section 5 — Conclusions

Write 3–5 sentences summarizing what you found. Address these questions:

- What is the most important thing your analysis revealed?
- What surprised you?
- What would you investigate next if you had more time or data?
- What are the limitations of this analysis — what can't you conclude from this data?

Then complete the competency claim below.

**Summary of findings:**  
Using PokéAPI data for the original 150 Pokémon, the analysis found evidence of a negative relationship between body size (height/weight) and speed, with meaningful variation across primary types. Primary type was also differentially associated with base stats, and the specialization index suggested that some types are more stat-concentrated while others are comparatively balanced. The iconicity analysis (using lower Pokédex ID as a proxy) provided a formal test of whether prominent early Pokémon differ in total base stats from later entries. Important limitations are that Pokédex order is an indirect proxy for recognizability and that the dataset is restricted to one generation's base stats, so conclusions should be interpreted as structural patterns rather than causal explanations of popularity.

---

## Competency Claim

In a `mp1.md` file in your GitHub repository, write a short competency claim (2–4 sentences) for each domain you feel this project demonstrates. Be specific — cite something you actually did in this notebook.

Domains covered by this project typically include:
- **C3 — Data cleaning and file handling** (if you cleaned or reshaped data)
- **C5 — Data analysis with pandas** (answering questions with code)
- **C6 — Data visualization** (your chart)
- **C7 — Critical evaluation and professional judgment** (your interpretation and limitations section)

You don't have to claim every domain — only the ones your work actually demonstrates.

---

## Competency Claim

In a `mp1.md` file in your GitHub repository, write a short competency claim (2–4 sentences) for each domain you feel this project demonstrates. Be specific — cite something you actually did in this notebook.

Domains covered by this project typically include:
- **C3 — Data cleaning and file handling** (if you cleaned or reshaped data)
- **C5 — Data analysis with pandas** (answering questions with code)
- **C6 — Data visualization** (your chart)
- **C7 — Critical evaluation and professional judgment** (your interpretation and limitations section)

You don't have to claim every domain — only the ones your work actually demonstrates.